# Fase 5: RUL Estimation - Deep Learning (GRU)

## Tujuan
Memprediksi Remaining Useful Life (RUL) mesin dalam satuan hari/jam menggunakan arsitektur GRU.

## Langkah-langkah
- 5.1: Rekayasa Target RUL (Hitung mundur dari titik failure, piece-wise cap 30 hari)
- 5.2: Transformasi Data ke Tensor 3D (Sliding Window, time_steps=24)
- 5.3: Arsitektur GRU (Eksperimen)
- 5.4: Training dengan Early Stopping
- 5.5: Evaluasi Regresi (MAE, RMSE) & Visualisasi RUL Aktual vs Prediksi
- 5.6: Ekspor Model Pemenang (rul_gru.h5)

In [1]:
import pandas as pd
import numpy as np

# 1. Muat data hasil tahap Feature Engineering
df = pd.read_csv('../data/processed/sensor_features_engineered.csv')

# 2. Pastikan kolom timestamp berformat datetime (belum menjadi index)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 3. Cari waktu maksimum (terakhir) saat mesin mati (failure == 1) untuk tiap mesin
# Buat referensi tabel terpisah menggunakan groupby, kemudian gabungkan
max_failure_times = df[df['failure'] == 1].groupby('machine_id')['timestamp'].max().reset_index()
max_failure_times = max_failure_times.rename(columns={'timestamp': 'max_failure_time'})

# Gabungkan max_failure_time ke Dataframe utama menggunakan left join
df = df.merge(max_failure_times, on='machine_id', how='left')

# 4. Hitung mundur Sisa Umur (RUL_days) 
# Menghitung selisih waktu dalam hari (menggunakan pd.Timedelta agar mendapatkan nilai desimal/float hari)
df['RUL_days'] = (df['max_failure_time'] - df['timestamp']) / pd.Timedelta(days=1)

# (Opsional tapi penting): Filter nilai negatif dari sensor anomali yang masih menyala jika mesin sudah diset mati
df = df[df['RUL_days'] >= 0]

# 5. Terapkan Trik Senior (Piece-wise RUL) limit maksimal 30 hari
# Potong semua nilai yang lebih dari 30 menjadi mentok 30 saja
df['RUL_days'] = df['RUL_days'].clip(upper=30)

# 6. Hapus baris yang berstatus NaN di kolom RUL_days (mesin yang belum pernah terdeteksi rusak)
df = df.dropna(subset=['RUL_days'])

# 7. Bersihkan kolom yang sudah tidak terpakai lagi
df = df.drop(columns=['failure', 'max_failure_time'])

# 8. Kembalikan timestamp menjadi index DataFrame
df = df.set_index('timestamp')

# 9. Cetak distribusi statistik akhir untuk memvalidasi Piece-wise RUL
print("=== Distribusi Target RUL_days (Piece-wise 30 Hari) ===")
print(df['RUL_days'].describe())


=== Distribusi Target RUL_days (Piece-wise 30 Hari) ===
count    57609.000000
mean        26.243559
std          7.815283
min          0.000000
25%         29.958333
50%         30.000000
75%         30.000000
max         30.000000
Name: RUL_days, dtype: float64


In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd

# Jika data awal perlu di-load ulang (opsional untuk memastikan kebersihan dari awal sel)
# df = pd.read_csv('../data/processed/sensor_features_engineered.csv')
# df['timestamp'] = pd.to_datetime(df['timestamp'])
# max_failure = df[df['failure'] == 1].groupby('machine_id')['timestamp'].max().reset_index().rename(columns={'timestamp': 'max_failure_time'})
# df = df.merge(max_failure, on='machine_id', how='left')
# df['RUL_days'] = (df['max_failure_time'] - df['timestamp']) / pd.Timedelta(days=1)
# df = df[df['RUL_days'] >= 0]
# df['RUL_days'] = df['RUL_days'].clip(upper=30)
# df = df.drop(columns=['failure', 'max_failure_time']).set_index('timestamp')

# --- 1. SAPU BERSIH NAN SILUMAN DARI ROTASI ---
df.dropna(inplace=True)

# SOLUSI: Pastikan X tidak memuat tipe teks string ('machine_id') atau string sisa lainnya
X = df.drop(columns=['RUL_days', 'machine_id'], errors='ignore') 
y = df['RUL_days']

# --- 2. LINEAR SPLIT 70:30 (HARAM ACAK/SHUFFLE) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# --- 3. HUKUM NORMALISASI FITUR (ANTI EXPLODING GRADIENT) ---
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. PEMBENTUKAN TENSOR 3D: (Samples, Time Steps, Features) ---
def create_sequences(X, y, time_steps=24):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 24
X_train_3D, y_train_seq = create_sequences(X_train_scaled, y_train.values)
X_test_3D, y_test_seq = create_sequences(X_test_scaled, y_test.values)

print(f"✅ Data Siap! Shape X_train_3D GRU: {X_train_3D.shape}")

# ==========================================================
# --- 5 & 6. BANGUN ARSITEKTUR GRU (Gated Recurrent Unit) ---
# ==========================================================
model_gru = Sequential()

# Pintu Masuk Data (Input Layer Keras 3.x)
model_gru.add(Input(shape=(X_train_3D.shape[1], X_train_3D.shape[2])))

# Layer Memori #1: GRU
model_gru.add(GRU(units=64, return_sequences=True))
model_gru.add(Dropout(0.2))

# Layer Memori #2: GRU
model_gru.add(GRU(units=32, return_sequences=False))

# Layer Output Regresi Mutlak
model_gru.add(Dense(units=1))

# --- 7 & 8. COMPILE & TRAINING GRU DENGAN REM DARURAT ---
model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("🚀 Memulai Pelatihan Model GRU Penantang...")
history_gru = model_gru.fit(
    X_train_3D, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_3D, y_test_seq),
    callbacks=[early_stop]
)


✅ Data Siap! Shape X_train_3D GRU: (39979, 24, 31)
🚀 Memulai Pelatihan Model GRU Penantang...
Epoch 1/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - loss: 140.2569 - mae: 9.8302 - val_loss: 142.8974 - val_mae: 8.6173
Epoch 2/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 28.8349 - mae: 2.9831 - val_loss: 152.8013 - val_mae: 8.6617
Epoch 3/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 28.7804 - mae: 2.8842 - val_loss: 151.2210 - val_mae: 8.6538
Epoch 4/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 28.7821 - mae: 2.8967 - val_loss: 154.8188 - val_mae: 8.6721
Epoch 5/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 28.7872 - mae: 2.8834 - val_loss: 150.7869 - val_mae: 8.6517
Epoch 6/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 28.7944 - mae: 2.8879 - val_loss: 153.4944 - val_mae: 8.6652
